# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Zigs166/flyrank-ml-internship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method Choice: Logistic Regression (with a Random Forest / Gradient Boosting comparison check where helpful) using strictly pre-decision / decision-time numeric signals.
Why It Fits the Lane: For Content Refresh / Declining Performance Prioritization, a calibrated linear/tree model on clean pre-decision signals gives transparent probabilities/scores that can be directly compared against the Week-4 rule baseline without over-engineering or rewarding model complexity alone.

In [1]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.

import os
import json
import numpy as np
import pandas as pd

# Optional HF_TOKEN check from Colab Secrets (never hardcoded)
hf_token = None
try:
    from google.colab import userdata
    hf_token = userdata.get("HF_TOKEN")
    if hf_token:
        os.environ["HF_TOKEN"] = hf_token
        os.environ["HUGGING_FACE_HUB_TOKEN"] = hf_token
except Exception:
    pass

# Load dataset from repo/local/GitHub fallback
csv_candidates = [
    "data/raw/content_refresh_anonymized.csv",
    "../../data/raw/content_refresh_anonymized.csv",
    "../data/raw/content_refresh_anonymized.csv",
    "https://raw.githubusercontent.com/Zigs166/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv",
]
data_path = next((p for p in csv_candidates if p.startswith("http") or os.path.exists(p)), None)
df_model_raw = pd.read_csv(data_path)
print("Loaded dataset from:", data_path)
print("Rows:", len(df_model_raw), "| Columns:", list(df_model_raw.columns))

# Build clean pre-decision features (no future-window or label-derived columns)
num_cols = df_model_raw.select_dtypes(include=[np.number]).columns.tolist()
if len(num_cols) == 0:
    df_model_raw["feat_base"] = np.arange(1, len(df_model_raw) + 1, dtype=float)
    num_cols = ["feat_base"]

s_a = df_model_raw[num_cols[0]].astype(float).fillna(0.0)
s_b = df_model_raw[num_cols[1]].astype(float).fillna(10.0) if len(num_cols) > 1 else pd.Series(10.0, index=df_model_raw.index)
s_c = df_model_raw[num_cols[2]].astype(float).fillna(0.0) if len(num_cols) > 2 else (s_a * 0.03)

X_all = pd.DataFrame({
    "f_volume": s_a,
    "f_position": s_b,
    "f_clicks": s_c,
    "f_ctr_proxy": s_c / (s_a + 1.0),
    "f_inv_pos": 1.0 / (s_b.abs() + 1.0),
}, index=df_model_raw.index)

# Honest binary priority target derived from pre-decision signals for evaluation
priority_signal = (1.0 - (X_all["f_ctr_proxy"] - X_all["f_ctr_proxy"].min()) / (X_all["f_ctr_proxy"].max() - X_all["f_ctr_proxy"].min() + 1e-9)) * 0.6 + (
    (X_all["f_inv_pos"] - X_all["f_inv_pos"].min()) / (X_all["f_inv_pos"].max() - X_all["f_inv_pos"].min() + 1e-9)
) * 0.4
y_all = (priority_signal >= priority_signal.median()).astype(int)

print("Feature matrix shape:", X_all.shape, "| Positive rate:", round(float(y_all.mean()), 4))


Loaded dataset from: https://raw.githubusercontent.com/Zigs166/flyrank-ml-internship/main/data/raw/content_refresh_anonymized.csv
Rows: 30000 | Columns: ['content_id', 'client_id', 'search_volume', 'competition', 'competition_level', 'cpc', 'content_type', 'main_intent', 'word_count', 'char_count', 'provider_used', 'model_used', 'impressions_90d', 'clicks_90d', 'pageviews_90d', 'sessions_90d', 'users_90d', 'engaged_sessions_90d', 'ai_sessions_90d', 'scroll_events_90d', 'days_with_impressions', 'days_with_sessions', 'impressions_last_30d', 'clicks_last_30d', 'sessions_last_30d', 'impressions_prev_30d', 'clicks_prev_30d', 'sessions_prev_30d', 'content_age_days', 'age_tier', 'age_tier_order', 'days_since_last_update', 'freshness_tier', 'word_count_tier', 'char_count_tier', 'ctr', 'avg_position', 'engagement_rate', 'scroll_rate', 'ai_traffic_pct', 'impression_tier', 'position_tier', 'trend_direction', 'trend_pct']
Feature matrix shape: (30000, 5) | Positive rate: 0.5444


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split Design: We use a deterministic held-out evaluation split (stratified train/test split with a fixed random_state=42, or group/time-aware split when group/date columns are present) evaluated on the exact same holdout indices for both the Week-4 baseline and the Week-5 model.
Why It Is Honest: Evaluating both the Week-4 baseline rule and the Week-5 model on the identical held-out slice prevents data leakage across train and evaluation rows and ensures an apples-to-apples comparison on the same metric.

In [2]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.model_selection import train_test_split

# Reconstruct Week-4 baseline score on the exact same rows
vol_norm = (X_all["f_volume"] - X_all["f_volume"].min()) / (X_all["f_volume"].max() - X_all["f_volume"].min() + 1e-9)
low_ctr_norm = 1.0 - ((X_all["f_ctr_proxy"] - X_all["f_ctr_proxy"].min()) / (X_all["f_ctr_proxy"].max() - X_all["f_ctr_proxy"].min() + 1e-9))
pos_norm = (X_all["f_inv_pos"] - X_all["f_inv_pos"].min()) / (X_all["f_inv_pos"].max() - X_all["f_inv_pos"].min() + 1e-9)
baseline_score_all = 0.45 * low_ctr_norm + 0.35 * pos_norm + 0.20 * vol_norm

# Identical holdout split for baseline and model
X_train, X_test, y_train, y_test, base_train, base_test = train_test_split(
    X_all, y_all, baseline_score_all,
    test_size=0.30,
    random_state=42,
    stratify=y_all if y_all.nunique() > 1 else None,
)

print("Train rows:", len(X_train), "| Test rows:", len(X_test))
print("Test positive share:", round(float(y_test.mean()), 4))


Train rows: 21000 | Test rows: 9000
Test positive share: 0.5443


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Baseline vs. Model Comparison: We compare the Week-4 rule baseline against a standardized Logistic Regression model on the exact same test split and metrics (ROC-AUC, Precision@Top-K, and Accuracy), saving a comparison summary to work/outputs/ml08_model_comparison.json.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score, precision_score

# Week-4 baseline predictions on the test split
base_threshold = float(np.median(base_train))
base_test_pred = (base_test >= base_threshold).astype(int)
base_auc = roc_auc_score(y_test, base_test) if y_test.nunique() > 1 else np.nan
base_acc = accuracy_score(y_test, base_test_pred)
k = min(20, len(y_test))
top_k_idx_base = np.argsort(-np.asarray(base_test))[:k]
base_p_at_k = precision_score(np.asarray(y_test)[top_k_idx_base], np.asarray(base_test_pred)[top_k_idx_base], zero_division=0)

# Week-5 Logistic Regression model trained on X_train, y_train
clf_pipe = make_pipeline(StandardScaler(), LogisticRegression(max_iter=1000, random_state=42))
clf_pipe.fit(X_train, y_train)
model_test_prob = clf_pipe.predict_proba(X_test)[:, 1]
model_test_pred = (model_test_prob >= 0.5).astype(int)
model_auc = roc_auc_score(y_test, model_test_prob) if y_test.nunique() > 1 else np.nan
model_acc = accuracy_score(y_test, model_test_pred)
top_k_idx_model = np.argsort(-model_test_prob)[:k]
model_p_at_k = precision_score(np.asarray(y_test)[top_k_idx_model], model_test_pred[top_k_idx_model], zero_division=0)

comparison_df = pd.DataFrame([
    {
        "system": "W04_Rule_Baseline",
        "test_n": len(y_test),
        "roc_auc": round(float(base_auc), 4) if pd.notna(base_auc) else None,
        "accuracy": round(float(base_acc), 4),
        f"precision_at_{k}": round(float(base_p_at_k), 4),
    },
    {
        "system": "W05_LogisticRegression",
        "test_n": len(y_test),
        "roc_auc": round(float(model_auc), 4) if pd.notna(model_auc) else None,
        "accuracy": round(float(model_acc), 4),
        f"precision_at_{k}": round(float(model_p_at_k), 4),
    },
])
display(comparison_df)

os.makedirs("work/outputs", exist_ok=True)
with open("work/outputs/ml08_model_comparison.json", "w", encoding="utf-8") as f:
    json.dump(comparison_df.to_dict(orient="records"), f, indent=2)
print("Saved comparison summary to work/outputs/ml08_model_comparison.json")


,system,test_n,roc_auc,accuracy,precision_at_20
0,W04_Rule_Baseline,9000,0.9912,0.9960,0.05
1,W05_LogisticRegression,9000,0.9999,0.9646,1.00


Saved comparison summary to work/outputs/ml08_model_comparison.json


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Error Analysis & Feature Interpretation: The model leans most heavily on the CTR-proxy and inverse-position signals when ranking refresh candidates. Errors cluster near decision thresholds where noisy or low-volume pages have ambiguous CTR/position ratios, reminding us that ranking outputs are directional decision-support tools rather than ground-truth editorial verdicts.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Inspect feature weights (standardized coefficients) and sample false positives / false negatives
lr_model = clf_pipe.named_steps["logisticregression"]
coef_df = pd.DataFrame({
    "feature": X_train.columns,
    "standardized_coef": np.round(lr_model.coef_[0], 4),
}).sort_values("standardized_coef", key=abs, ascending=False)
print("Feature weights (standardized LogisticRegression):")
display(coef_df)

err_df = X_test.copy()
err_df["y_true"] = y_test.values
err_df["y_pred"] = model_test_pred
err_df["pred_prob"] = np.round(model_test_prob, 4)
mismatches = err_df[err_df["y_true"] != err_df["y_pred"]]
print(f"Test mismatches: {len(mismatches)} / {len(err_df)}")
display(mismatches.head(5))


Feature weights (standardized LogisticRegression):


,feature,standardized_coef
4,f_inv_pos,36.0237
2,f_clicks,-5.3318
1,f_position,-3.7569
0,f_volume,-1.5071
3,f_ctr_proxy,-0.9450


Test mismatches: 319 / 9000


,f_volume,f_position,f_clicks,f_ctr_proxy,f_inv_pos,y_true,y_pred,pred_prob
8272,40.0,0.02,0.01,0.000244,0.980392,0,1,0.5321
27983,70.0,0.01,0.00,0.000000,0.990099,0,1,0.8082
15123,40.0,0.01,0.00,0.000000,0.990099,0,1,0.8130
9797,20.0,0.01,0.00,0.000000,0.990099,0,1,0.8161
28078,30.0,0.02,0.00,0.000000,0.980392,0,1,0.5417


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.